In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import sklearn.ensemble
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import itertools 
import sklearn
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib
pd.set_option('future.no_silent_downcasting', True)



In [11]:
# to install xgboost on the notebook environment
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [12]:
def convert_min_to_float(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        mins, secs = map(int, min_str.split(':'))
        return mins + secs / 60
    return 0.0  # handle empty or malformed entries

In [13]:
def convert_int_season_to_str(season):
    if isinstance(season, int):
        return f"{season}-{season%2000 +1 :02d}" 
    return season

In [14]:
# make me a function that will print the number of unique values in each column of the dataframe
def print_unique_values(df):
    for column in df.columns:
        print(f"{column}: {len(df[column].unique())} unique {column}")


In [15]:
NUM_GAMES=82
teams=['DAL','MIL','ATL','DEN','HOU','IND','OKC','CHI','ORL','BOS','DET','NYK'
,'CHA','LAL','SAC','MIA','LAC','GSW','POR','MIN','WAS','BKN','MEM','SAS'
,'PHX','NOP','UTA','TOR','PHI','CLE']
all_possible_matchups=itertools.combinations(teams, 2)
regular_games_total=pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_totals_2010_2024.csv",delimiter=',',header=0)
regular_season_all_parts=pd.concat([
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_1.csv",delimiter=',',header=0),
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_2.csv",delimiter=',',header=0),
        pd.read_csv("./datasets/NBA_DATA_2010_2024/regular_season_box_scores_2010_2024_part_3.csv",delimiter=',',header=0)])

In [16]:
def getMatchAndPlayerStats(game,player,season=None,teamname=None):
    """
    Function to get the average points of a team in a season
    :param teamname: team name :List[str]
    :param season: season: str or int or None (for all seasons) 
    :return: average points of the team 
    """
    season=convert_int_season_to_str(season)
    playerScores = player[player['minutes'].notna()].copy()
    playerScores['minutesParsed'] = playerScores['minutes'].apply(convert_min_to_float)
    game.loc[:,'WL'] = game['WL'].replace({'W': 1, 'L': 0}).infer_objects(copy=False)
    gamePlayer=game.merge(playerScores, how='inner', left_on=['GAME_ID','TEAM_ABBREVIATION'], right_on=['gameId','teamTricode'])
    # add a collumn to count the number of games played by each player
    aggregation= gamePlayer.groupby(['personName','teamTricode','season_year']).agg(
        {
            'WL': 'sum',
            'plusMinusPoints':'mean',
            'minutesParsed': 'mean',
            'points': 'mean',
            'fieldGoalsPercentage': 'mean',
            'threePointersPercentage': 'mean',
            'reboundsTotal': 'mean',
            'foulsPersonal': 'mean',
            'turnovers': 'mean',
            'fieldGoalsMade': 'mean',
            'fieldGoalsAttempted': 'mean',
            'steals':'mean',
        }
    ).reset_index()
    aggregation['gamesPlayed'] = gamePlayer.groupby(['personName','teamTricode','season_year'])['gameId'].count().reset_index(drop=True)

    if season is not None:
        aggregation = aggregation[aggregation['season_year'] == season]
    if teamname is not None:
        aggregation = aggregation[aggregation['teamTricode'].isin(teamname)]
    aggregation['winPercentage'] = aggregation['WL'] / aggregation['gamesPlayed'] 
    return aggregation.reset_index(drop=True)

In [19]:
gameP = getMatchAndPlayerStats(regular_games_total, regular_season_all_parts)

# Features e target
X = gameP.drop(['personName','teamTricode','season_year',"plusMinusPoints",'winPercentage'], axis=1)
y = gameP['plusMinusPoints']

# Standardização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 1. Modelo Vanilla
vanilla_model = XGBRegressor(n_estimators=100)
vanilla_model.fit(X_train, y_train)

y_pred_vanilla = vanilla_model.predict(X_test)
r2_vanilla = r2_score(y_test, y_pred_vanilla)
mse_vanilla = mean_squared_error(y_test, y_pred_vanilla)

print("Vanilla XGBoost:")
print(f"R2:  {r2_vanilla:.4f}")
print(f"MSE: {mse_vanilla:.4f}\n")


# 2. Modelo com Tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=XGBRegressor(),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Melhor modelo
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

r2_tuned = r2_score(y_test, y_pred_tuned)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)

print("Tuned XGBoost:")
print("Best params:", grid_search.best_params_)
print(f"R2:  {r2_tuned:.4f}")
print(f"MSE: {mse_tuned:.4f}")

best_model.save_model('./models/xgb_tunned.json')

Vanilla XGBoost:
R2:  0.4304
MSE: 5.3947

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Tuned XGBoost:
Best params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
R2:  0.4839
MSE: 4.8878


In [18]:
# RANDOM FOREST REGRESSOR
# Vanilla Random Forest
rf_vanilla = RandomForestRegressor(n_estimators=100, random_state=42)
rf_vanilla.fit(X_train, y_train)

y_pred_vanilla_rf = rf_vanilla.predict(X_test)
r2_vanilla_rf = r2_score(y_test, y_pred_vanilla_rf)
mse_vanilla_rf = mean_squared_error(y_test, y_pred_vanilla_rf)

print("Vanilla Random Forest:")
print(f"R2:  {r2_vanilla_rf:.4f}")
print(f"MSE: {mse_vanilla_rf:.4f}\n")

# Modelo base
rf = RandomForestRegressor(random_state=42)

# Grid de hiperparâmetros a testar
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 25, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [2,5],
}

# GridSearch com validação cruzada
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# Treinar com dados (X_train e y_train devem já estar prontos e normalizados se necessário)
grid_search_rf.fit(X_train, y_train)

# Melhor modelo
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)


print("Tuned Random Forest:")
print("Best params:", grid_search_rf.best_params_)
print("R2:", r2_score(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))

Vanilla Random Forest:
R2:  0.4871
MSE: 4.8573

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Tuned Random Forest:
Best params: {'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}
R2: 0.49548527439671397
MSE: 4.778042488645394
